# **HADDOCK3 Tutorial on Protein-Glycan Modelling**

# **Introduction**

This tutorial demonstrates the use of HADDOCK3 for predicting the structure of a protein-glycan complex using information about the protein binding site.

A glycan is a molecule composed of different monosaccharide units, linked to each other by glycosidic bonds. Glycans are involved in a wide range of biological processes, such as cell-cell recognition, cell adhesion, and immune response. Glycan are highly diverse and complex in their structure, as they can involve multiple *branches* and different *linkages*, namely different ways in which a glycosidic bond can connect two monosaccharides. This complexity together with their flexibility makes the prediction of glycan-protein interactions a challenging task.

In this tutorial we will be working with the catalytic domain of the *Humicola Grisea* Cel12A enzyme (PDB code [1OLR](https://www.ebi.ac.uk/pdbe/entry/pdb/1olr) )and a linear homopolymer, *4-beta-glucopyranose*, as glycan (PDB code of the complex [1UU6](https://www.ebi.ac.uk/pdbe/entry/pdb/1uu6)).

The tutorial is based on [A. Ranaudo et al., J. Chem. Inf. Model. 64 (19), 7816-7825, 2024](https://pubs.acs.org/doi/10.1021/acs.jcim.4c01372).

<figure width=75% style="text-align: center;">
  <img src="https://www.bonvinlab.org/education/HADDOCK3/HADDOCK3-protein-glycan/1UU6.png">
</figure>
                    
<figure style="text-align: center">
    <i>Picture of the protein-glycan complex in pdb 1UU6</i>

---
# **A brief introduction to HADDOCK3**

HADDOCK3 is the next generation integrative modelling software in the
long-lasting HADDOCK project. It represents a complete rethinking and rewriting
of the HADDOCK2.X series, implementing a new way to interact with HADDOCK and
offering new features to users who can now define custom workflows.

In the previous HADDOCK2.x versions, users had access to a highly
parameterisable yet rigid simulation pipeline composed of three steps:
`rigid-body docking (it0)`, `semi-flexible refinement (it1)`, and `final refinement (itw)`.

<figure style="text-align: center;">
<img width="75%" src="https://bonvinlab.org/education/HADDOCK3/HADDOCK3-antibody-antigen/HADDOCK2-stages.png">
</figure>

In HADDOCK3, users have the freedom to configure docking workflows into
functional pipelines by combining the different HADDOCK3 modules, thus
adapting the workflows to their projects. HADDOCK3 has therefore developed to
truthfully work like a puzzle of many pieces (simulation modules) that users combine freely. To this end, the old HADDOCK machinery has been modularized,
and several new modules added, including third-party software additions. As a
result, the modularization achieved in HADDOCK3 allows users to duplicate steps
within one workflow (e.g., to repeat twice the `it1` stage of the HADDOCK2.x
rigid workflow).

Note that, for simplification purposes, at this time, not all functionalities of
HADDOCK2.x have been ported to HADDOCK3, which does not (yet) support NMR RDC,
PCS and diffusion anisotropy restraints, cryo-EM restraints and coarse-graining.
Any type of information that can be converted into ambiguous interaction
restraints can, however, be used in HADDOCK3, which also supports the
*ab initio* docking modes of HADDOCK.

<figure style="text-align: center;">
<img width="75%" src="https://bonvinlab.org/education/HADDOCK3/HADDOCK3-antibody-antigen/HADDOCK3-workflow-scheme.png">
</figure>

To keep HADDOCK3 modules organized, we catalogued them into several
categories. However, there are no constraints on piping modules of different
categories.

The main module categories are "topology", "sampling", "refinement",
"scoring", and "analysis". There is no limit to how many modules can belong to a
category. Modules are added as developed, and new categories will be created
if/when needed. You can access the HADDOCK3 documentation page for the list of
all categories and modules. Below is a summary of the available modules:

* **Topology modules**
    * `topoaa`: *generates the all-atom topologies for the CNS engine.*
* **Sampling modules**
    * `rigidbody`: *Rigid body energy minimization with CNS (`it0` in haddock2.x).*
    * `lightdock`: *Third-party glow-worm swam optimization docking software.*
* **Model refinement modules**
    * `flexref`: *Semi-flexible refinement using a simulated annealing protocol through molecular dynamics simulations in torsion angle space (`it1` in haddock2.x).*
    * `emref`: *Refinement by energy minimisation (`itw` EM only in haddock2.4).*
    * `mdref`: *Refinement by a short molecular dynamics simulation in explicit solvent (`itw` in haddock2.X).*
* **Scoring modules**
    * `emscoring`: *scoring of a complex performing a short EM (builds the topology and all missing atoms).*
    * `mdscoring`: *scoring of a complex performing a short MD in explicit solvent + EM (builds the topology and all missing atoms).*
* **Analysis modules**
    * `alascan`: *Performs a systematic (or user-define) alanine scanning mutagenesis of interface residues.*
    * `caprieval`: *Calculates CAPRI metrics (i-RMSD, l-RMSD, Fnat, DockQ) with respect to the top-scoring model or reference structure if provided.*
    * `clustfcc`: *Clusters models based on the fraction of common contacts (FCC)*
    * `clustrmsd`: *Clusters models based on pairwise RMSD matrix calculated with the `rmsdmatrix` module.*
    * `contactmap`: *Generate contact matrices of both intra- and intermolecular contacts and a chordchart of intermolecular contacts.*
    * `rmsdmatrix`: *Calculates the pairwise RMSD matrix between all the models generated in the previous step.*
    * `ilrmsdmatrix`: *Calculates the pairwise interface-ligand-RMSD (il-RMSD) matrix between all the models generated in the previous step.*
    * `seletop`: *Selects the top N models from the previous step.*
    * `seletopclusts`: *Selects the top N clusters from the previous step.*

The HADDOCK3 workflows are defined in simple configuration text files, similar to the TOML format but with extra features.
Contrary to HADDOCK2.X which follows a rigid (yet highly parameterisable)
procedure, in HADDOCK3, you can create your own simulation workflows by
combining a multitude of independent modules that perform specialized tasks.


---
# **Software and data setup**

In order to follow this tutorial we will start Jupyter session in Colab or other resources, install haddock3 and download the data for the tutorial.

To run this tutorial on Colab we recommand changing the runtime type to use one of the TPU option, which will give you access to more cores. The haddock3 docking run described in this tutorial should complete in about 15 minutes.

For this follow the following steps(comment out the `pip install haddock3` line if you already have a working installation):

In [ ]:
#@title 1. Install haddock3, BioPython and py3Dmol
#@markdown Execute this cell to download the required packages.
!pip install haddock3 --quiet
!pip install py3Dmol --quiet
!pip install BioPython --quiet

In [ ]:
#@title 2. Download the tutorial data
#@markdown We are providing pre-processed PDB files for docking and analysis (but the preprocessing of those files will also be explained in this tutorial). The files have been processed to facilitate their use in HADDOCK and to allow comparison with the known reference structure of the complex. Execute this cell to download and decompress the data.
!wget https://surfdrive.surf.nl/public.php/dav/files/GoEKHFSLZa43gCY -O HADDOCK3-protein-glycan-notebook.zip
!unzip -oq HADDOCK3-protein-glycan-notebook.zip
!\rm HADDOCK3-protein-glycan-notebook.zip
import os
from pathlib import Path
BASE_DIR = Path.cwd()
PROJECT_NAME = 'HADDOCK3-protein-glycan-notebook'
PROJECT_DIR = BASE_DIR / PROJECT_NAME
os.chdir(PROJECT_DIR)

Unziping the file will create the HADDOCK3-protein-glycan directory which should contain the following directories and files:

* `pdbs`: a directory containing the pre-processed PDB files
* `restraints`: a directory containing the interface information and the corresponding restraint files for HADDOCK3
* `runs`: a directory containing pre-calculated results
* `scripts`: a directory containing various scripts used in this tutorial
* `workflows`: Contains HADDOCK3 configuration files for the various scenarios in this 


---
# **Preparing PDB files for docking**

In this section we will prepare the PDB files of the protein and glycan for docking.

Ready-to-dock structures are available in `pdbs` directory.

## **Preparing the protein structure**

A crystal structure of the protein in the unbound form is available (1OLR) from the [PDB](https://www.rcsb.org/structure/1OLR), which we will be using for docking.

The following command fetches the PDB ID and removes water and heteroatoms (in this case no co-factor is present that should be kept).


In [ ]:
! pdb_fetch 1OLR | pdb_tidy -strict | pdb_delhetatm | pdb_keepcoord | pdb_tidy -strict > 1OLR_ready.pdb


## **Building the glycan structure using the GLYCAM server** (optional)

We can model the glycan using the [GLYCAM webserver](https://glycam.org/cb/). Our glycan is a linear polymer consisting of 4 beta-D-glucopyranose units. Beta-D-glucopyranose is a common monosaccharide found basically in all the living organisms. In this case the four monosaccharides are linked by beta-1,4-glycosidic bonds, where the *anomeric carbon* (C1) of one monosaccharide is linked to the C4 of the next one.

We can start by accessing the [GLYCAM webserver](https://glycam.org/cb/) where we will model the glycan! 

__*Select the Glc monosaccharide with your mouse.*__

This will add the monosaccharide to the sequence, together with an OH group as *aglycon* (the non-sugar part of a glycan).

Now we need to add a second beta-D-glucopyranose unit to the sequence. We will link it to the first one by a beta-1,4-glycosidic bond.

__*Click one more time on the Glc icon.*__

Now you are asked to specify a linkage.
 __*Select the Beta linkage and the 1-4 option. Do not change the other parameters.*__

This will add the second monosaccharide to the sequence, linked to the first one by a beta-1,4-glycosidic bond.

__*Repeat the process to add the third and fourth monosaccharide units to the sequence.*__

At the end of the process you should observe the following sequence:

__DGlcpb1-4DGlcpb1-4DGlcpb1-4DGlcpa1-OH__

__*Press the Done button and wait for the webserver to process your request. When done, press Download Minimized Structure and download the PDB file.*__

__*Note*__ that the GLYCAM-generated model can be found in the `pdbs` directory as `DGlcpb1-4DGlcpb1-4DGlcpb1-4DGlcpb1-OH_structure.pdb`



## **Adapting the GLYCAM-generated model for HADDOCK use** (optional)


Unfortunately, the glycan structure we just obtained cannot be directly used in HADDOCK as the format and the residue and atom naming differ from the conventions used in HADDOCK (which follow the naming in the PDB). We will need to edit it to remove the several TER statements GLYCAM placed between the monosaccharides, and to add the [HADDOCK residue name](https://wenmr.science.uu.nl/haddock2.4/library) proper to beta-D-glucopyranose. Importantly, we have to merge the OH aglycon with the first monosaccharide unit, as they are now separated in two different residues.

__*What is the HADDOCK three letter code corresponding to beta-D-glucopyranose?*__
<details style="background-color:#DAE4E7">
  <summary style="bold">
      <b><i>Answer</i></b>
  </summary>
  <figure style="text-align: justified;">
  <p>
  Looking at the <a href="https://wenmr.science.uu.nl/haddock2.4/library">https://wenmr.science.uu.nl/haddock2.4/library</a> website, beta-D-glucopyranose is identified by BGC.
  </p>
</details>
<br>

Let us start with the aglycon. The next command will select the GLYCAM residue name proper to the aglycon, changes the chain ID to B, and changes the residue name to the HADDOCK one. The resulting structure is saved in the aglycon.pdb file.

In [ ]:
!pdb_selresname -ROH pdbs/DGlcpb1-4DGlcpb1-4DGlcpb1-4DGlcpb1-OH_structure.pdb | pdb_tidy -strict | pdb_chain -B | pdb_rplresname -ROH:BGC > aglycon.pdb

Now we will process the remaining of the glycan structure. 
 
The `pdb_tidy` command removes the TER statements between each unit, while `pdb_selres` selects all the residues except the OH aglycon. The `pdb_chain` command changes the chain ID to B, and the `pdb_rplresname` command changes the residue name to BGC. The last command, `pdb_reres`, renumbers the residues of the glycan starting from 1.

In [ ]:
!pdb_tidy -strict pdbs/DGlcpb1-4DGlcpb1-4DGlcpb1-4DGlcpb1-OH_structure.pdb | pdb_selres -2:5 | pdb_chain -B | pdb_rplresname -0GB:BGC | pdb_rplresname -4GB:BGC | pdb_reres -1 > sugar.pdb


Finally we will merge the two structures in the 1UU6_l_u.pdb file:

In [ ]:
!pdb_merge aglycon.pdb sugar.pdb | pdb_tidy > 1UU6_l_u.pdb


__*Note*__ that the pre-processed glycan structure can be found in the `pdbs` directory of the archive you downloaded.



## **How good is the generated GLYCAM model?**

We will now check how close is the modelled glycan to the reference structure. For this we will use py3Dmol to superimpose the two structures and calculate the RMSD (using the ring atoms). Execute the following cell:

In [ ]:
from haddock.libs.libnotebooks import align_full
import py3Dmol
import gzip
import os

# Create a 3Dmol viewer object
view = py3Dmol.view(width=800, height=600)

path1 = PROJECT_DIR / "pdbs" / "1UU6_l_u.pdb"
path2 = PROJECT_DIR / "pdbs" / "1uu6_chain_B.pdb"

# Align the matching chain B from both structures
view = align_full(str(path1),str(path2),chains=["B"],atom_types=["C1", "C2", "C3", "C4", "C5", "O1"],)

# First structure: orange sticks
view.addStyle({"model": 0, "chain": "B"},{"stick": {"radius": 0.20},})
view.addStyle({"model": 0, "elem": "C", "chain": "B"},{"stick": {"color": "orange","radius": 0.20},})

# Second structure: green sticks
view.addStyle({"model": 1, "chain": "B"},{"stick": {"radius": 0.20},})
view.addStyle({"model": 1, "elem": "C", "chain": "B"},{"stick": {"color": "green","radius": 0.20},})

# Zoom specifically onto chain B
view.zoomTo({"chain": "B"})

view.show()

__*What is the RMSD between the two glycan structures? In which of the four monosaccharide units is the model accurate? In which ones is it not?*__
<details style="background-color:#DAE4E7">
  <summary style="bold">
      <b><i>Answer</i></b>
  </summary>
  <figure style="text-align: justified;">
  <p>
  The reported RMSD calculated using the ring atoms of each glycan unit is 2.74Å. Looking at the alignment, the first two glycans units are matching reasonably well, while more deviations are observed in the last two.
  </p>
</details>
<br>


---
# **Defining restraints for docking**

##  **Visualising the information about the binding site**

Here we mimic a scenario where we have information about the glycan binding site on the protein, but no knowledge about which monosaccharide units are relevant for the binding. In this case (see Fig. 1), all the four beta-D-glucopyranose units are at the interface, although this might not be true in general, especially when longer glycans are considered.

The following residues correspond to the protein binding site, as calculated from the crystal structure of the complex are:

`22,24,59,64,97,103,105,115,120,122,124,131,132,133,134,155,158,205,207`
Let us visualize the interface on our unbound protein structure. For this start PyMol and load the PDB file of the unbound protein:



In [ ]:
import py3Dmol
import gzip
import os

view = py3Dmol.view(width=800, height=600)

pdb_file_path = "./pdbs/1OLR_clean.pdb"

if not os.path.exists(pdb_file_path):
    print(f"Error: File not found at {pdb_file_path}")

else:
    if pdb_file_path.endswith(".gz"):
        with gzip.open(pdb_file_path, "rt") as file:
            pdb_data = file.read()
    else:
        with open(pdb_file_path, "r") as file:
            pdb_data = file.read()

    view.addModel(pdb_data, "pdb")

    view.setStyle({'cartoon': {'color': 'white'}})
    view.addStyle({'stick': {'color': 'white'}})
    view.addStyle({'resi': [22, 24, 59, 64, 97, 103, 105, 115, 120, 122, 124, 131, 132, 133, 134, 155, 158, 205, 207]},{'cartoon':{'color':'red'}})
    view.addStyle({'resi': [22, 24, 59, 64, 97, 103, 105, 115, 120, 122, 124, 131, 132, 133, 134, 155, 158, 205, 207]},{'stick':{'color':'red'}})
    view.addSurface(py3Dmol.SAS, {'opacity':0.4, 'color':'white'})
    view.zoomTo()
    view.show()    


---
## **Generating the restraints**

In this section we will define the restraints that will guide the docking of the protein and glycan structures.

A description of the format for the various restraint types supported by HADDOCK can be found in our [Nature Protocol](https://www.nature.com/articles/s41596-024-01011-0.epdf?sharing_token=UHDrW9bNh3BqijxD2u9Xd9RgN0jAjWel9jnR3ZoTv0O8Cyf_B_3QikVaNIBRHxp9xyFsQ7dSV3t-kBtpCaFZWPfnuUnAtvRG_vkef9o4oWuhrOLGbBXJVlaaA9ALOULn6NjxbiqC2VkmpD2ZR_r-o0sgRZoHVz10JqIYOeus_nM%3D) paper, Box 1. Information about various types of distance restraints in HADDOCK can also be found in our [online manual](https://www.bonvinlab.org/software/haddock2.4/airs/) pages.

We will use `haddock3-restraints` to generate the restraints for the protein-glycan docking. For this we need to define two files, one for each molecule, containing on the first line the list of `active residues` (those that should be at the interface) and on the second line the list of `passive residues` (those that can be at the interface

For the protein, we will only define active residues (based on the identified binding site above) and leave the second line (passive) empty, while for the glycan we only define passive residues in the second line, leaving the first line empty. The corresponding files are provided as:

* *restraints/1olr-binding-site.act* for the protein
* *restraints/glycan.pass* for the glycan

The command to generate a HADDOCK ambiguous distance restraint file (.tbl) from those file is:


In [ ]:
!haddock3-restraints active_passive_to_ambig restraints/1olr-binding-site.act restraints/glycan.pass > ambig.tbl

We can check the validity of the generated tbl file (useful when manually editing restraint files) with:

In [ ]:
!haddock3-restraints validate_tbl ambig.tbl

__*Inspect the generated file. Can you understand the syntax?*__

Distance restraints in the HADDOCK/CNS syntax are defined as:

`assign (selection1) (selection2) distance, lower-bound correction, upper-bound correction`
where the lower limit for the distance is calculated as: `distance minus lower-bound correction` and the upper limit as: `distance plus upper-bound correction`.

__*What is the distance range (lower and upper limits) defined for the restraints?*__
<details style="background-color:#DAE4E7">
  <summary style="bold">
      <b><i>Answer</i></b>
  </summary>
  <figure style="text-align: justified;">
  <p>
   Each restraint (<i>assign</i></figure> statements) has as distance and lower/upper bound limits respectively 2.0 2.0 0.0. This means that the lower distance limit (lower bound) is 0.0Å and the upper distance limit is 2.0Å. 2.0Å is actually shorter than the minimum van der Waals distance between two heavy atoms, which might sound strange.  The effective distance for a restraint is calculated from many pairwise distances (the combination of all atoms selected by the two selections between brackets); it is defined as deff = [Sum 1/d**6)]**1/6. This function causes the effective distance to always be shorter than the shortest distance entering the sum.
  </p>
</details>
<br>


---
# **Setting up the docking with HADDOCK3**

Now that we have all required files at hand (PDB and restraints files) it is time to setup our docking protocol. For this we need to create a HADDOCK3 configuration file that will define the docking workflow. We will illustrate this flexibility by introducing a clustering step after the initial rigid-body docking stage, select up to 20 models per cluster and refine all of those.

HADDOCK3 also provides an analysis module (`caprieval`) that allows to compare models to either the best scoring model (if no reference is given) or to a reference structure, which in our case we have at hand.

The basic workflow for all three scenarios will consists of the following modules, with some differences in the restraints used and some parameter settings (see below):

* __topoaa__: *Generates the topologies for the CNS engine and build missing atoms*
* __rigidbody__: *Rigid body energy minimisation (it0 in haddock2.x)*
* __caprieval__: *Calculates CAPRI metrics (i-RMSD, l-RMSD, Fnat, DockQ) with respect to the top scoring model or reference structure if provided*
* __ilrmsdmatrix__: *Calculate the ilRMSD matrix between the generated models*
* __clustrmsd__: *Cluster the ilRMSD matrix in 10 clusters*
* __seletopclusts__: *Selection of the top20 models of each clusters*
* __caprieval__
* __flexref__: *Semi-flexible refinement of the interface (it1 in haddock2.4)*
* __caprieval__
* __ilrmsdmatrix__: *Calculate the ilRMSD matrix between the generated models*
* __clustrmsd__: *Cluster the ilRMSD matrix* now based on a RMSD distance cutoff
* __seletopclusts__: *Selection of the top4 models of all clusters*
* __caprieval__
* __contactmap__: *Generate a contact map for each cluster*

The configuration file for this scenario is already provided in the `workflows`directory of the archive you downloaded as `protein-glycans.cfg` (limited sampling for this tutorial) and `protein-glycan-full.cfg` (full sampling).

```toml
# ====================================================================
# protein-glycan docking using information about the protein binding site
# and no information on the glycan side
# ====================================================================

# directory in which the docking will be done
run_dir = "run_prot-glycan"

# compute mode
mode = "local"
ncores = 10

#input molecules
molecules = [
    "pdbs/1OLR_clean.pdb",
    "pdbs/1UU6_l_u.pdb",
] 

# ====================================================================
# Parameters for each stage are defined below, prefer full paths
# ====================================================================
[topoaa] 

[rigidbody]
# Reduced sampling (200 instead of the default of 1000)
sampling = 200
# Protein binding site to glycan ambig restraints
ambig_fname = "restraints/ambig.tbl"
# Increased vdw weight for the HADDOCK score
w_vdw = 1

[caprieval]
reference_fname = "pdbs/1UU6_target.pdb"

[ilrmsdmatrix]

[clustrmsd]
criterion = 'maxclust'
# Number of clusters to be formed
n_clusters = 10

[seletopclusts]
# Reduced number of models per cluster (full run uses 20)
top_models = 5

[caprieval]
reference_fname = "pdbs/1UU6_target.pdb"

[flexref]
# Protein binding site to glycan ambig restraints
ambig_fname = "restraints/ambig.tbl"

[caprieval]
reference_fname = "pdbs/1UU6_target.pdb"

[ilrmsdmatrix]

[clustrmsd]
criterion = 'distance'
linkage = 'average'
min_population = 4
clust_cutoff = 2.5 

[seletopclusts]
top_models = 4

[caprieval]
reference_fname = "pdbs/1UU6_target.pdb"

[contactmap]

# ====================================================================```

__Important for glycans__: the weight of the van der Waals component of the HADDOCK score at the rigidbody stage has been set to 1.0 rather than the default 0.01 (`vdw_w = 1.0`). This is in agreement with the settings used for protein-small molecules docking in HADDOCK2.4, as explained in [Journal of Computer-Aided Molecular Design, 2019](https://link.springer.com/article/10.1007/s10822-019-00244-6) and also applied to protein-glycan docking.

__Note__ the clustering step placed between the rigidbody and the flexible refinement stages. The idea here is that the scoring function at the rigidbody level is not perfect, thus good models may not be selected if we only consider the top 200 models (as in HADDOCK2.X). Clustering the rigidbody models allows to increase the diversity of the models selected for the flexible refinement stage. In this example, we request the models to be clustered into 10 clusters using the `maxclust` option. The second `clustrmsd` step after flexible refinement uses a different clustering methods based on RMSD between models using a 2.5Å cutoff. 


# **Running HADDOCK3**

In the first section of the configuration file you can see the definition of the global parameters:

```toml
# compute mode
mode = "local"
ncores = 10
```
The parameter `mode` defines how this workflow will be executed. In this case, it will run locally, on your machine, using up to 10 CPUs. Feel free to change this value (edit for that the `runs/protein-glycan.cfg` file which we are using), if more cores are available.

As everything is ready, you can run the workflow by executing the next cell. 

__*Using 10 cores on a Max OSX M2 processor the tutorial workflow executes in ~4 minutes while the full workflow would take ~31 min.*__

If you do not wish to wait for the run to finish, you can directly skip to the analysis section as we will be using the precomputed data from the `runs` directory.

In [ ]:
!haddock3 workflows/protein-glycan.cfg

---
# **Analysis of docking results**

---
## **Inspecting the results of the docking run**

Once your run has completed inspect the content of the resulting directory. You will find the various steps (modules) of the defined workflow numbered sequentially, e.g.:

```toml
> ls run_prot-glycans/
  00_topoaa
  01_rigidbody
  02_caprieval
  03_ilrmsdmatrix
  04_clustrmsd
  05_seletopclusts
  06_caprieval
  07_flexref
  08_caprieval
  09_ilrmsdmatrix
  10_clustrmsd
  11_seletopclusts
  12_caprieval
  13_contactmap
  analysis
  data
  log
  traceback
```

In the following, for this tutorial, we will inspect the results of a pre-calculated full sampling run, which can be found in `runs/run_protein-glycan-full`.

There is in addition to the various modules defined in the config workflow a log file (text file) and three additional directories:

* the `data` directory containing the input data (PDB and restraint files) for the various modules
* the `analysis` directory containing various plots to visualise the results for each `caprieval` step
* the `traceback` directory containing the names of the generated models for each step, allowing to trace back a model throughout the various stages.

You can find information about the duration of the run at the bottom of the log file. Each sampling/refinement/selection module will contain PDB files

For example, the `11_seletopclusts` directory contains the selected models from each cluster. The clusters in that directory are numbered based on their rank, i.e. `cluster_1` refers to the top-ranked cluster. Information about the origin of these files can be found in that directory in the `seletopclusts.txt` file.

The simplest way to extract ranking information and the corresponding HADDOCK scores is to look at the `X_caprieval`directories (which is why it is a good idea to have it as the final module, and possibly as intermediate steps, even when no reference structures are known). This directory will always contain a `capri_ss.tsv` file, which contains the model names, rankings and statistics (score, iRMSD, Fnat, lRMSD, ilRMSD and dockq score). E.g. from `runs/protein-glycan-full/08_caprieval` (the analysis step right after the `flexref` stage:



```
../07_flexref/flexref_127.pdb	-	1	-180.605	1.223	0.864	3.249	3.253	0.779	0.783	-	-	-	7.569	354.067	62.826	1085.740	0.000	0.000	0.000	-4.390	1173.930	-115.358	102.968	0.000	0.000	0.000	-158.545	-50.756	0.000	0.000
../07_flexref/flexref_92.pdb	-	2	-170.394	1.224	0.795	3.256	3.263	0.756	0.785	-	-	-	6.977	353.362	56.474	1082.110	0.000	0.000	0.000	-4.851	1177.660	-108.312	103.165	0.000	0.000	0.000	-148.443	-47.108	0.000	0.000
../07_flexref/flexref_226.pdb	-	3	-169.150	1.316	0.682	3.583	3.576	0.699	0.839	-	-	-	42.410	364.496	63.827	1047.770	0.000	0.000	0.000	-4.239	1178.980	-130.772	107.024	0.000	0.000	0.000	-116.265	-27.902	0.000	0.000
../07_flexref/flexref_44.pdb	-	4	-168.426	1.141	0.750	3.058	3.059	0.756	0.744	-	-	-	1.215	350.469	60.991	1044.670	0.000	0.000	0.000	-5.366	1177.140	-115.858	104.652	0.000	0.000	0.000	-151.520	-36.877	0.000	0.000
../07_flexref/flexref_159.pdb	-	5	-160.324	1.260	0.636	3.454	3.464	0.694	0.808	-	-	-	22.623	354.969	62.209	1039.850	0.000	0.000	0.000	-3.523	1173.150	-109.922	105.459	0.000	0.000	0.000	-126.042	-38.743	0.000	0.000
../07_flexref/flexref_170.pdb	-	6	-155.609	1.224	0.636	3.334	3.335	0.701	0.783	-	-	-	27.275	350.079	62.061	1067.000	0.000	0.000	0.000	-2.779	1174.790	-103.550	104.117	0.000	0.000	0.000	-117.613	-41.338	0.000	0.000
...
```

The iRMSD, lRMSD and Fnat metrics are the ones used in the blind protein-protein prediction experiment CAPRI (Critical PRediction of Interactions).

In CAPRI the quality of a model is defined as (for protein-protein complexes):

* **Unacceptable**:    i-RMSD >4Å or Fnat < 0.1 (DOCKQ < 0.23)
* **acceptable model**: i-RMSD < 4Å or l-RMSD < 10Å and Fnat > 0.1 (0.23 < DOCKQ < 0.49)
* **medium quality model**: i-RMSD < 2Å or l-RMSD < 5Å and Fnat > 0.3 (0.49 < DOCKQ < 0.8)
* **high quality model**: i-RMSD < 1Å or l-RMSD < 1Å and Fnat > 0.5 (DOCKQ > 0.8)

As these metrics are for protein-protein complexes and glycans are typically smaller, it is best to use stricter metrics to assess the quality of the models. In the case of information-driven protein-glycan docking, the Fnat term is less relevant, as most contacts will typically be satisfied.

For protein-glycan modelling we recently proposed a different, stricter metric based on the __*interface ligand RMSD (ilRMSD)*__, see [A. Ranaudo et al., J. Chem. Inf. Model. 64 (19), 7816-7825, 2024](https://pubs.acs.org/doi/10.1021/acs.jcim.4c01372):

* __near acceptable model__: ilRMSD < 4Å (0.35 < DOCKQ < 0.43)
* __acceptable model__: ilRMSD < 3Å (0.43 < DOCKQ < 0.715)
* __medium quality model__: ilRMSD < 2Å (0.71 < DOCKQ < 0.895)
* __high quality model__: ilRMSD < 1Å (DOCKQ > 0.895)

The ilRMSD is calculated by fitting the models onto the refence using the interface residues of the receptor (the protein in this case) and calculated the RMSD on the ligand (the glycans in this case).

__*What is based on this criterion the quality of the top ranked model listed above (flexref_127.pdb)?*__ 
<details style="background-color:#DAE4E7">
  <summary style="bold">
      <b><i>Answer</i></b>
  </summary>
  <figure style="text-align: justified;">
  <p>
   Based on the DOCKQ values it would be an acceptable quality model (although the ilRMSD values puts it in the near acceptable category).
  </p>
</details>
<br>

__*Consider now the model with the best ilRMSD (lowest value). Which model is it and what is its quality?*__

The ilRMSD values are listed in column 8 of the capri_ss.tsv file. To find the lowest ilRMSD value you can sort the file numerically based on column 8 and extract the top model. This can be done at the command line with the following command:

In [ ]:
!sort -nk8 runs/run_prot-glycan-full/08_caprieval/capri_ss.tsv | head -2

<br>
<details style="background-color:#DAE4E7">
  <summary style="bold">
      <b><i>Answer</i></b>
  </summary>
  <figure style="text-align: justified;">
  <p>
   The best model is model flexref_48, ranked at position 21, with a DOCKQ of 0.763 and an ilRMSD of 2.904Å. Based on DOCKQ this would be a medium quality model.
  </p>
</details>
<br>


## Impact of the flexible refinement


Since we have `caprieval` steps at various stages of the workflow we can assess the impact of the flexible refinement.

To facilate this analysis we are providing a script that extracts CAPRI stats from each `caprieval` step. Again we will use the results of the pre-calculated full sampling run, which can be found in `runs/run_protein-glycan-full`.

To extract the statistics execute the following cell:



In [ ]:
!./scripts//extract-capri-stats-dockq-glycans.sh runs/run_prot-glycan-full

<br>

__*Check first the total number of acceptable models before (`06_caprieval`) and after (`08_caprieval`) flexible refinement. How is this number changing?*__
<details style="background-color:#DAE4E7">
  <summary style="bold">
      <b><i>Answer</i></b>
  </summary>
  <figure style="text-align: justified;">
  <p>
   We observe an increase in the number of acceptable models from 18 to 53 after flexible refinement, indicating the refinement does improve the quality of the models.
  </p>
</details>
<br>

__*Look now the best model metrics (il-RMSD, DOCKQ). What is the impact of the flexible refinement stage on those?*__
<details style="background-color:#DAE4E7">
  <summary style="bold">
      <b><i>Answer</i></b>
  </summary>
  <figure style="text-align: justified;">
  <p>
   Here again we observe quite an improvement in quality, both in terms of il-RMSD and DOCKQ. The same can be observed for the first-acceptable model.
  </p>
</details>
<br>


---

# **Cluster-based analysis**

In case where the `caprieval` module is called after a clustering step, an additional `capri_clt.tsv` file will be present in the directory.
This file contains the cluster ranking and score statistics, averaged over the minimum number of models defined for clustering
(4 by default), with their corresponding standard deviations. E.g. for `runs/run_prot-glycan-full/12_caprieval`:

```
cluster_rank	cluster_id	n	under_eval	score	score_std	irmsd	irmsd_std	fnat	fnat_std	lrmsd	lrmsd_std	dockq	dockq_std	ilrmsd	ilrmsd_std	rmsd	rmsd_std	air	air_std	bsa	bsa_std	desolv	desolv_std	elec	elec_std	total	total_std	vdw	vdw_std	caprieval_rank
1	4	4	-	-163.377	5.667	1.235	0.064	0.676	0.047	3.357	0.194	0.712	0.025	3.358	0.193	0.793	0.035	23.381	14.741	1049.822	10.311	-3.977	0.954	-115.026	10.079	-127.860	14.165	-36.215	5.054	1
2	1	4	-	-143.930	6.920	4.536	0.061	0.312	0.019	12.838	0.179	0.239	0.008	12.826	0.179	2.813	0.039	44.965	6.111	903.009	38.142	-2.154	2.386	-98.536	12.276	-92.277	13.773	-38.707	3.442	2
3	3	4	-	-127.024	12.147	2.214	0.125	0.392	0.019	6.457	0.391	0.447	0.023	6.444	0.387	1.414	0.081	46.742	9.419	883.878	32.337	-4.545	1.088	-85.937	14.440	-71.571	20.680	-32.377	2.223	3
4	6	4	-	-125.309	15.347	4.162	0.054	0.250	0.016	11.640	0.177	0.238	0.005	11.635	0.176	2.571	0.035	9.370	11.520	1027.426	37.250	-4.755	2.135	-59.533	21.569	-101.847	23.070	-51.684	10.707	4
5	2	4	-	-123.081	5.276	1.683	0.173	0.506	0.034	4.855	0.519	0.569	0.041	4.838	0.522	1.082	0.108	35.282	13.220	947.179	29.131	-5.684	2.268	-66.978	7.109	-76.171	14.972	-44.476	3.442	5
6	5	4	-	-73.883	12.966	4.086	0.087	0.131	0.010	11.701	0.253	0.198	0.006	11.708	0.253	2.554	0.054	80.649	25.131	677.865	35.566	1.253	1.661	-53.672	15.666	4.226	30.371	-22.750	3.565	6
```

In this file you find the cluster rank (which corresponds to the naming of the clusters in the previous `seletop` directory), the cluster ID (which is related to the size of the cluster, 1 being always the largest cluster), the number of models (n) in the cluster and the corresponding statistics (averages + standard deviations). The corresponding cluster PDB files will be found in the preceeding `09_seletopclusts` directory.

In this example all clusters have a size of 4 because we set `top_models=4` in the `seletopclusts` module. The real size of a cluster can be found in `10_clustrmsd/clustrmsd.txt`.

While these simple text files can be easily checked from the command line already, they might be cumbersome to read.
For that reason, we have developed a post-processing analysis that automatically generates html reports for all `caprieval` steps in the workflow.
These are located in the respective `analysis/XX_caprieval` directories and can be viewed using your favorite web browser. We will inspect those in the following.



---

# **Visualisation of HADDOCK score and their components**

Let us now analyse the docking results. Use for that either your own run or a pre-calculated run provided in the runs directory. The analysis/12_caprieval_analysis directory of the respective run directory contains various html files which can be visualised in a web browser. The report.html contains interactive plots that may take some time to generate.

For this tutorial we will visualise parts of this file directly in the notebook.


## **Cluster statistics**
On the top of the page, you will see a table that summarises the cluster statistics (taken from the capri_clt.tsv file). The columns (corresponding to the various clusters) are sorted by default on the cluster rank, which is based on the HADDOCK score.

For the sake of this tutorial we will analyse the results from the full sampling run provided in `runs/run_prot-glycan-full`.

__*Examine the plots (remember here that higher DockQ values and lower il-RMSD values correspond to better models)*__

In [ ]:
#@title View the cluster statistics table
import json
import re
from plotly.offline import iplot
from IPython.display import display
import pandas as pd

# Specify the path to your HTML file
html_file_path = PROJECT_DIR / "runs" / "run_prot-glycan-full" / "analysis" / "12_caprieval_analysis/report.html"

# Read the HTML file
with open(html_file_path, 'r') as f:
    html_content = f.read()

# Use a regular expression to find the script tag with id="datatable2" and extract its content
match_table = re.search(r'<script id="datatable2" type="application/json">\s*(.*?)\s*</script>', html_content, re.DOTALL)

if match_table:
    json_data_str_table = match_table.group(1)
    # Load the JSON data
    table_data = json.loads(json_data_str_table)

    # Extract the 'clusters' list from the JSON data
    clusters_data = table_data.get('clusters', [])

    # Create a pandas DataFrame from the clusters data
    df = pd.json_normalize(clusters_data)

    # Drop the columns related to best cluster files
    cols_to_drop = ['under_eval', 'total', 'rmsd', 'best1', 'best2', 'best3', 'best4', 'irmsd.mean', 'irmsd.std', 'lrmsd.mean', 'lrmsd.std', 'air.mean', 'air.std', 'desolv.mean', 'desolv.std', 'elec.mean', 'elec.std', 'vdw.mean', 'vdw.std'] # Add more if there are more 'best' columns
    df = df.drop(columns=[col for col in cols_to_drop if col in df.columns])

    # Display the DataFrame
    display(df)
else:
    print("Could not find the script tag with id='datatable2' in the HTML file.")


Inspect the final cluster statistics.

__*How many clusters have been generated?*__
<details style="background-color:#DAE4E7">
  <summary style="bold">
      <b><i>Answer</i></b>
  </summary>
  <figure style="text-align: justified;">
  <p>
  There are at least 10 clusters generated (the others are gathered under the "Other" row.
  </p>
</details>
<br>

__*Look at the score of the first few clusters: Are they significantly different if you consider their average scores and standard deviations?*__
<details style="background-color:#DAE4E7">
  <summary style="bold">
      <b><i>Answer</i></b>
  </summary>
  <figure style="text-align: justified;">
  <p>
  Cluster 1 is significantly better in score that the second-ranked cluster. Their average scores do not overlap when considering the standard deviations. This is not the case for clusters 2 and 3 for example.
  </p>
</details>
<br>

Since for this tutorial we have at hand the crystal structure of the complex, we provided it as reference to the `caprieval` modules.
This means that the ilRMSD, Fnat and DockQ statistics report on the quality of the docked model compared to the reference crystal structure. Remember that high DockQ and FCC values, along with low RMSD values, indicate better model quality.

* __near acceptable model__: ilRMSD < 4Å (0.35 < DOCKQ < 0.43)
* __acceptable model__: ilRMSD < 3Å (0.43 < DOCKQ < 0.715)
* __medium quality model__: ilRMSD < 2Å (0.71 < DOCKQ < 0.895)
* __high quality model__: ilRMSD < 1Å (DOCKQ > 0.895)


__*How many clusters of acceptable or better quality have been generate according to CAPRI criteria?*__
<details style="background-color:#DAE4E7">
  <summary style="bold">
      <b><i>Answer</i></b>
  </summary>
  <figure style="text-align: justified;">
  <p>
  Looking at the DockQ values, six clusters have DockQ values >0.43: Clusters 1, 3, 4, 6, 7 and 10.
  </p>
</details>
<br>

__*What is the rank of the best cluster generated?*__
<details style="background-color:#DAE4E7">
  <summary style="bold">
      <b><i>Answer</i></b>
  </summary>
  <figure style="text-align: justified;">
  <p>
  This is cluster1, i.e. the top-ranked one.
  </p>
</details>
<br>


## **Visualisation of the HADDOCK scores and their components**

Below the cluster statistics table, you’ll find a series of plots displaying the HADDOCK score and its components against various metrics (i-RMSD, l-RMSD, FCC, il-RMSD, DockQ), with clusters represented using color coding. The last rows show plots of cluster statistics, i.e. distributions of values per cluster, ordered by their HADDOCK score.

These plots are interactive. A menu will appear at the top right, just above the last plot in the first row, when you hover your mouse over it. This menu allows you to zoom in and out of the plots and toggle the visibility of clusters.

To view theses plots in the notebook execute the following cell.

__*Examine the plots (remember here that higher DockQ values and lower il-RMSD values correspond to better models)*__


In [ ]:
#@title View the various statistics plots
import json
import re
from IPython.display import display
import pandas as pd

# Specify the path to your HTML file
html_file_path = PROJECT_DIR  / "runs" / "run_prot-glycan-full" / "analysis" / "12_caprieval_analysis/report.html"

# Read the HTML file
with open(html_file_path, 'r') as f:
    html_content = f.read()

# Use a regular expression to find the script tag with id="data1" and extract its content
match_plot = re.search(r'<script id="data1" type="application/json">\s*(.*?)\s*</script>', html_content, re.DOTALL)

if match_plot:
    json_data_str_plot = match_plot.group(1)
    # Load the JSON data
    plot_data = json.loads(json_data_str_plot)

    # Set width and height to None to make the plot responsive
    if 'layout' in plot_data:
        plot_data['layout']['width'] = None
        #plot_data['layout']['height'] = None

    # Display the plot using iplot
    iplot(plot_data)
else:
    print("Could not find the script tag with id='data1' in the HTML file.")

<br>

__*Look at the plots of the HADDOCK score and various components against the il-RMSD. How well does the HADDOCK scoring funtion performs in this case?*__
<details style="background-color:#DAE4E7">
  <summary style="bold">
      <b><i>Answer</i></b>
  </summary>
  <figure style="text-align: justified;">
  <p>
  There is no clear correlation between il-RMSD and the HADDOCK score, but the scoring does rank the best quality cluster on top.
  </p>
</details>
<br>



Depending on the docking models, there could be a set of unclustered models. It will be explicitly shown in report.html as ‘other’. You can see the origins of these models in traceback/traceback.tsv.

Finally, the report also shows plots of the cluster statistics (distributions of values per cluster ordered according to their HADDOCK rank).

To view those distribution plots execute the next cell.

In [ ]:
#@title View the various distribution plots
import json
import re
from plotly.offline import iplot

html_file_path = PROJECT_DIR / "runs" / "run_prot-glycan-full" / "analysis" / "12_caprieval_analysis" / "report.html"

with open(html_file_path, 'r') as f:
    html_content = f.read()

match_plot = re.search(r'<script id="data2" type="application/json">\s*(.*?)\s*</script>', html_content, re.DOTALL)

if match_plot:
    json_data_str_plot = match_plot.group(1)
    plot_data = json.loads(json_data_str_plot)

    if 'layout' in plot_data:
        plot_data['layout']['width'] = None
        # Remove the template entirely — it contains deprecated trace types
        # (e.g. heatmapgl) that newer plotly versions no longer recognize
        plot_data['layout'].pop('template', None)

    iplot(plot_data, validate=False)
else:
    print("Could not find the script tag with id='data1' in the HTML file.")

---
# **Contact Analysis**

We have added a contact analysis module to HADDOCK3 that generates for each cluster both a contact matrix of the entire system showing all contacts within a 4.5Å cutoff and a chord chart representation of intermolecular contacts.

In the current workflow we run, those files can be found in the `13_contactmap` directory. These are again html files with interactive plots (hover with your mouse over the plots).

This file taken from the pre-computed run can be visualized by executing the next cell which will visualize the contacts of the first ranked cluster (cluster 1).

__*Can you identify which residue(s) make(s) the most intermolecular contacts?*__



In [ ]:
#@title View the contact chord chart
import json
import re
from plotly.offline import iplot
from IPython.display import display

# Specify the path to the HTML file containing the chord chart
html_file_path = PROJECT_DIR / "runs" / "run_prot-glycan-full" / "13_contactmap" / "cluster1_chordchart.html"

# Read the HTML file
with open(html_file_path, 'r') as f:
    html_content = f.read()

# Use a regular expression to find the script tag containing the plot data
# The id of the script tag might vary, so a more general pattern is used
# We assume the plot data is within a script tag with type="application/json"
match_plot = re.search(r'<script.*?type="application/json".*?>\s*(.*?)\s*</script>', html_content, re.DOTALL)

if match_plot:
    json_data_str_plot = match_plot.group(1)
    # Load the JSON data
    plot_data = json.loads(json_data_str_plot)
    # Display the plot using iplot
    iplot(plot_data)
else:
    print(f"Could not find the plot data in the HTML file at {html_file_path}")

---
# **Visualisation of the docking models**

It’s time to visualise some of the docking models. Let’s take a look at `cluster_1_model_1.pdb.gz`, the best-ranked model and the reference structure `pdbs/1UU6_target.pdb`.

We will align the best model onto the crystal structure of the complex

In [ ]:
from haddock.libs.libnotebooks import align_full
import py3Dmol
import gzip
import os

# Create a 3Dmol viewer object
view = py3Dmol.view(width=800, height=600)

path1 = PROJECT_DIR / "pdbs" / "1UU6_target.pdb"
path2 = PROJECT_DIR / "runs" / "run_prot-glycan-full" / "11_seletopclusts" / "cluster_1_model_1.pdb.gz"

# Align the matching chain B from both structures
view = align_full(str(path1),str(path2),chains=["A"],)

# First structure glycan: orange sticks
view.addStyle({"model": 0, "chain": "B"},{"stick": {"radius": 0.20},})
view.addStyle({"model": 0, "elem": "C", "chain": "B"},{"stick": {"color": "orange","radius": 0.20},})

# Second structure glycan: green sticks
view.addStyle({"model": 1, "chain": "B"},{"stick": {"radius": 0.20},})
view.addStyle({"model": 1, "elem": "C", "chain": "B"},{"stick": {"color": "green","radius": 0.20},})

view.show()

---
# **Conclusions**

In this tutorial we have demonstrated the use of HADDOCK3 to predict the structure of a protein-glycan complex using information about the protein binding site, starting from the apo structure of the receptor and a built model of the glycan. We have shown how to prepare the PDB files for docking, define the restraints, and set up the docking protocol. We have also discussed the analysis of the docking results and the comparison with the reference structure.

Below you will find an additional bonus section answering the question wether docking from an ensemble of glycan conformations improve the results? It also explains how to generate this ensemble of conformations

We hope you have enjoyed this tutorial and that you have learned something new. If you have any questions or feedback, please do not hesitate to contact us on the [HADDOCK forum](https://ask.bioexcel.eu/c/haddock/6).

---
---
# **BONUS1: Using an ensemble of glycan conformations**

## **Creating an ensemble of glycan conformations**

In this tutorial we have modelled a single conformation of the glycan using the GLYCAM server. However, glycans are highly flexible molecules that can adopt multiple conformations. Indeed, the modelled glycan is quite different from the conformation adopted in the reference structure. To account for this flexibility, we can generate an ensemble of glycan structures that will be docked to the protein.

To do this, we will use the short Molecular Dynamics refinement protocol available in HADDOCK3. This protocol will generate an ensemble of glycan conformations by running short MD simulations in explicit solvent (water) on the glycan structure. The hope here is to observe a glycan structure that is significantly closer to the structure observed in the complex (bound conformation). For a more thorough and proper sampling optimized MD software such as e.g. [GROMACS](https://www.gromacs.org/) and [OPENMM](https://openmm.org/), are better suited. but for the sake of this tutorial, we will use HADDOCK3.

We will use a protocol, starting from the GLYCAM conformation, consisting of the following steps:

* __topoaa__: *Generates the topologies for the CNS engine and build missing atoms*
* __mdref__: *MD refinement in explicit solven*
* __rmsdmatrix__: *Calculate the RMSD matrix between the generated models*
* __clustrmsd__: *Cluster the RMSD matrix using a distance criterion*
* __seletopclusts__: *Select one model per cluster*

This protocol will run a short MD simulation on the glycan structure, generate an ensemble of conformations, cluster them based on their RMSD using a 0.6Å cutoff, and select the cluster center from each cluster.

__Note__ how the glycan is here defined as fully flexible (`nfle1 = 1`). The number of steps of the different mdref parameters has also been increased with respect to the default values to ensure a better sampling of the conformational space. The sampling factor has been set to 50 to generate 50 conformations.

__*If you have sufficient computing power try to increase the sampling factor to 400.*__

It is available as `glycan-mdref.cfg` in the `workflow` directory of the archive you downloaded:

```toml
# ====================================================================
# MD Refinment of the glycan conformation
# ====================================================================

# directory in which the docking will be done
run_dir = "run_glycan-mdref"

# compute mode
mode = "local"
ncores = 10

# starting glycan conformation
molecules =  [
    "pdbs/1UU6_l_u.pdb",
    ]

# ====================================================================
# Parameters for each stage are defined below, prefer full paths
# ====================================================================
[topoaa]

[mdref]
# give full flexibility ro the glycan
nfle1 = 1
fle_sta_1_1 = 1
fle_end_1_1 = 4
# generate 50 models
sampling_factor = 100
# increase number of MD steps for sampling
watersteps = 50000
watercoolsteps = 10000

[rmsdmatrix]

[clustrmsd]
criterion = 'distance'
linkage = 'average'
min_population = 1
clust_cutoff = 0.6

[seletopclusts]
top_models = 1

# ====================================================================
```


To run the protocol above, go into the `haddock3` directory and execute the following command:


In [ ]:
!haddock3 workflows/glycan-mdref.cfg 


This will generate a new directory `run-glycan-mdref` with the results of the MD refinement. On a Mac M2, this run completes in ~4 1/2 minutes.

We are interested in the content of the 4_selectopclusts directory, which contains the selected representative model from each cluster. In this case 6 cluster representatives were generated.

We will first create an ensemble of models that contains the original model plus the MD-sampled ones.


In [ ]:
!gzip -d runs/run_glycan-mdref/0_topoaa/*pdb.gz
!gzip -d runs/run_glycan-mdref/4_seletopclusts/cluster*pdb.gz
!pdb_mkensemble runs/run_glycan-mdref/0_topoaa/1UU6_l_u_haddock.pdb runs/run_glycan-mdref/4_seletopclusts/cluster*pdb | pdb_tidy > 1UU6_l_u_ens.pdb



We can now visualise those models and compare them to the bound glycan structure (in orange).


In [ ]:
from haddock.libs.libnotebooks import align_full_ens
import py3Dmol
import gzip
import os

ref_path = PROJECT_DIR / "pdbs" / "1UU6_l_b.pdb"          # reference (single model)
ensemble_path = PROJECT_DIR / "pdbs" / "1UU6_l_u_ens.pdb"  # multi-model ensemble

# Align every model of the ensemble onto the reference using chain B
view = align_full_ens(
    str(ref_path),
    str(ensemble_path),
    animate=True,
    show_model_number=True,
    chains=["B"],
    atom_types=["C1", "C2", "C3", "C4", "C5", "O1"],
)

# Ensemble models (all models except the reference) as green sticks.
# addStyle without a "model" selector applies to every model...
view.addStyle({"chain": "B"}, {"stick": {"radius": 0.20}})
view.addStyle({"elem": "C", "chain": "B"}, {"stick": {"color": "green", "radius": 0.20}})

# ...then override the reference (model 0) with orange sticks.
view.addStyle({"model": 0, "chain": "B"}, {"stick": {"radius": 0.20}})
view.addStyle({"model": 0, "elem": "C", "chain": "B"}, {"stick": {"color": "orange", "radius": 0.20}})

# Zoom specifically onto chain B
view.zoomTo({"chain": "B"})

view.show()

__*Look at the RMSD values. Has the MD sampling generated models closer to the bound form?*__

---
## **Docking from an ensemble of glycan conformations**

To do this, we will use the `protein-glycan-ens.cfg`, available in the `haddock3` directory of the archive you downloaded.
Besides the presence of the glycan ensemble (`1UU6_l_u_ens`) in place of the single structure (`1UU6_l_u`, the only difference between this protocol and the previous one is the `sampling` parameter at the rigidbody docking level: as multiple conformations are being docked, in order to sample each starting conformation a sufficient amount of time we increase the number of models generated. Here is the workflow for a full sampling run.

```toml
# ====================================================================
# protein-glycan docking using information about the protein binding site
# and no information on the glycan side
# ====================================================================

# directory in which the docking will be done
run_dir = "run_prot-glycan-ens-full"

# compute mode
mode = "local"
ncores = 10

#input molecules
molecules = [
    "pdbs/1OLR_clean.pdb",
    "pdbs/1UU6_l_u_ens.pdb",
] 

# ====================================================================
# Parameters for each stage are defined below, prefer full paths
# ====================================================================
[topoaa] 

[rigidbody]
# Increased sampling because of ensemble (2000 instead of the default of 1000)
sampling = 2000
# Protein binding site to glycan ambig restraints
ambig_fname = "restraints/ambig.tbl"
# Increased vdw weight for the HADDOCK score
w_vdw = 1

[caprieval]
reference_fname = "pdbs/1UU6_target.pdb"

[ilrmsdmatrix]

[clustrmsd]
criterion = 'maxclust'
# Number of clusters to be formed
n_clusters = 50

[seletopclusts]
top_models = 20

[caprieval]
reference_fname = "pdbs/1UU6_target.pdb"

[flexref]
# Protein binding site to glycan ambig restraints
ambig_fname = "restraints/ambig.tbl"

[caprieval]
reference_fname = "pdbs/1UU6_target.pdb"

[ilrmsdmatrix]

[clustrmsd]
criterion = 'distance'
linkage = 'average'
min_population = 2
clust_cutoff = 2.5 

[seletopclusts]
top_models = 4

[caprieval]
reference_fname = "pdbs/1UU6_target.pdb"

[contactmap]

# ====================================================================
```


This full sampling workflow completes in about 31 min. on a Mac M2. If you want to run it here, uncomment (remove the "#") from the next cell and execute it.


In [ ]:
#!haddock3 workflows/protein-glycan-ens-full.cfg



For the purpose of answering the question if using an ensemble of conformations improve the docking results we will use pre-calculated data from `runs/run_prot-glycan-ens-full`

Let's now have a look at the cluster statistics for this ensemble docking run.



In [ ]:
#@title View the cluster statistics table
import json
import re
from plotly.offline import iplot
from IPython.display import display
import pandas as pd

# Specify the path to your HTML file
html_file_path = PROJECT_DIR / "runs" / "run_prot-glycan-ens-full" / "analysis" / "12_caprieval_analysis/report.html"

# Read the HTML file
with open(html_file_path, 'r') as f:
    html_content = f.read()

# Use a regular expression to find the script tag with id="datatable2" and extract its content
match_table = re.search(r'<script id="datatable2" type="application/json">\s*(.*?)\s*</script>', html_content, re.DOTALL)

if match_table:
    json_data_str_table = match_table.group(1)
    # Load the JSON data
    table_data = json.loads(json_data_str_table)

    # Extract the 'clusters' list from the JSON data
    clusters_data = table_data.get('clusters', [])

    # Create a pandas DataFrame from the clusters data
    df = pd.json_normalize(clusters_data)

    # Drop the columns related to best cluster files
    cols_to_drop = ['under_eval', 'total', 'rmsd', 'best1', 'best2', 'best3', 'best4', 'irmsd.mean', 'irmsd.std', 'lrmsd.mean', 'lrmsd.std', 'air.mean', 'air.std', 'desolv.mean', 'desolv.std', 'elec.mean', 'elec.std', 'vdw.mean', 'vdw.std'] # Add more if there are more 'best' columns
    df = df.drop(columns=[col for col in cols_to_drop if col in df.columns])

    # Display the DataFrame
    display(df)
else:
    print("Could not find the script tag with id='datatable2' in the HTML file.")


For comparison here is the same statistics table for the run from the single model generated with the GLYCAM server.

In [ ]:
#@title View the cluster statistics table
import json
import re
from plotly.offline import iplot
from IPython.display import display
import pandas as pd

# Specify the path to your HTML file
html_file_path = PROJECT_DIR / "runs" / "run_prot-glycan-full" / "analysis" / "12_caprieval_analysis/report.html"

# Read the HTML file
with open(html_file_path, 'r') as f:
    html_content = f.read()

# Use a regular expression to find the script tag with id="datatable2" and extract its content
match_table = re.search(r'<script id="datatable2" type="application/json">\s*(.*?)\s*</script>', html_content, re.DOTALL)

if match_table:
    json_data_str_table = match_table.group(1)
    # Load the JSON data
    table_data = json.loads(json_data_str_table)

    # Extract the 'clusters' list from the JSON data
    clusters_data = table_data.get('clusters', [])

    # Create a pandas DataFrame from the clusters data
    df = pd.json_normalize(clusters_data)

    # Drop the columns related to best cluster files
    cols_to_drop = ['under_eval', 'total', 'rmsd', 'best1', 'best2', 'best3', 'best4', 'irmsd.mean', 'irmsd.std', 'lrmsd.mean', 'lrmsd.std', 'air.mean', 'air.std', 'desolv.mean', 'desolv.std', 'elec.mean', 'elec.std', 'vdw.mean', 'vdw.std'] # Add more if there are more 'best' columns
    df = df.drop(columns=[col for col in cols_to_drop if col in df.columns])

    # Display the DataFrame
    display(df)
else:
    print("Could not find the script tag with id='datatable2' in the HTML file.")


In both runs the top-ranked cluster is also the best in terms of quality with the highest DocqQ score.

__*Compare the il-RMSD and DockQ metrics of the ensemble and single model runs. Did using an ensemble of model improve the docking results?*__
<details style="background-color:#DAE4E7">
  <summary style="bold">
      <b><i>Answer</i></b>
  </summary>
  <figure style="text-align: justified;">
  <p>
  Using the pre-sampled ensemble of glycan conformations indeed improved the docking results. The quality of the top-ranked and best cluster improved with average DockQ value of 0.84 vs 0.71 for the single model run. The same applies to the average il-RMSD with decreased from 3.6 to 2.18. 
  </p>
</details>
<br>



__*Consider now the model with the best ilRMSD (lowest value). Which model is it and what is its quality?*__

The ilRMSD values are listed in column 8 of the capri_ss.tsv file. To find the lowest ilRMSD value you can sort the file numerically based on column 8 and extract the top model. This can be done at the command line with the following command:


In [ ]:
!sort -nk8 runs/run_prot-glycan-ens-full/08_caprieval/capri_ss.tsv | head -2

<br>
<details style="background-color:#DAE4E7">
  <summary style="bold">
      <b><i>Answer</i></b>
  </summary>
  <figure style="text-align: justified;">
  <p>
   The best model is model flexref_18, ranked at position 1, with a DOCKQ of 0.875 and an ilRMSD of 1.956Å. Based on DOCKQ and il-RMSD this is a medium quality model. It is also the top-ranked model of the top-ranked cluster.
  </p>
</details>
<br>

Let's visualise `cluster_1_model_1.pdb.gz`, the best-ranked model and the reference structure `pdbs/1UU6_target.pdb`.

We will align the best model onto the crystal structure of the complex.


In [ ]:
from haddock.libs.libnotebooks import align_full
import py3Dmol
import gzip
import os

# Create a 3Dmol viewer object
view = py3Dmol.view(width=800, height=600)

path1 = PROJECT_DIR / "pdbs" / "1UU6_target.pdb"
path2 = PROJECT_DIR / "runs" / "run_prot-glycan-ens-full" / "11_seletopclusts" / "cluster_1_model_1.pdb.gz"

# Align the matching chain B from both structures
view = align_full(str(path1),str(path2),chains=["A"],)

# First structure glycan: orange sticks
view.addStyle({"model": 0, "chain": "B"},{"stick": {"radius": 0.20},})
view.addStyle({"model": 0, "elem": "C", "chain": "B"},{"stick": {"color": "orange","radius": 0.20},})

# Second structure glycan: green sticks
view.addStyle({"model": 1, "chain": "B"},{"stick": {"radius": 0.20},})
view.addStyle({"model": 1, "elem": "C", "chain": "B"},{"stick": {"color": "green","radius": 0.20},})

view.show()


Compared to the best model of the single run analysed in the main section, the glycan conformation in the binding site is much closer to the crystal structure with the main deviation in a terminal glycan unit.


---

---

# **Save your results to Google Drive**

In order to save the entire tutorial directory with your results run the following cell.

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define the source directory
source_dir = './HADDOCK3-protein-glycan-notebook'

# Define the destination directory in Google Drive
destination_dir = '/content/drive/MyDrive/'

# Create the destination directory if it doesn't exist
os.makedirs(destination_dir, exist_ok=True)

# Move the directory
!cd ../; mv "$source_dir" "$destination_dir"

print(f"Moved '{source_dir}' to '{destination_dir}'")